# UGRID GIS Operations

This notebook demonstrates the GIS analysis methods available on
`UgridDataset`. These methods follow the kernel + facade pattern:
standalone numpy/scipy kernels in focused modules, wired through
the `_UgridFacades` mixin.

We build a small synthetic mesh so every cell runs fast and
the examples are self-contained (no external data files needed).

In [ ]:
%matplotlib inline

import numpy as np

from pyramids.netcdf.ugrid import UgridDataset

## 1. Build a Synthetic Mesh

We create a simple 4×4 quad mesh (16 faces) with synthetic
bathymetry data so every example below has something to work with.

In [ ]:
nx, ny = 5, 5
x = np.linspace(0, 400, nx)
y = np.linspace(0, 400, ny)
xx, yy = np.meshgrid(x, y)
node_x = xx.ravel().astype(np.float64)
node_y = yy.ravel().astype(np.float64)

faces = []
for j in range(ny - 1):
    for i in range(nx - 1):
        bl = j * nx + i
        br = bl + 1
        tl = bl + nx
        tr = tl + 1
        faces.append([bl, br, tr, tl])
fnc = np.array(faces, dtype=np.intp)

depth = np.array(
    [1.0, 2.0, 3.0, 4.0, 2.0, np.nan, 5.0, 6.0, 3.0, 4.0, 7.0, 8.0, 4.0, 5.0, 6.0, 9.0],
    dtype=np.float64,
)

ds = UgridDataset.from_arrays(
    node_x=node_x,
    node_y=node_y,
    face_node_connectivity=fnc,
    data={"depth": depth},
    data_locations={"depth": "face"},
    epsg=28992,
    mesh_name="mesh2d",
)
print(ds)
print(f"Faces: {ds.n_face}, Nodes: {ds.n_node}, CRS: {ds.epsg}")

## 2. CRS Management

### set_crs — assign / override a CRS without reprojecting

`from_arrays` defaults to EPSG:4326 (WGS 84). If the coordinates
are actually in a local grid, use `set_crs` to correct the CRS
declaration without touching the coordinates.

In [ ]:
ds_default_crs = UgridDataset.from_arrays(
    node_x=node_x,
    node_y=node_y,
    face_node_connectivity=fnc,
    data={"depth": depth},
    data_locations={"depth": "face"},
    mesh_name="mesh2d",
)
print(f"Before (default): CRS = EPSG:{ds_default_crs.epsg}")

ds_default_crs.set_crs(epsg=28992)
print(f"After set_crs:    CRS = EPSG:{ds_default_crs.epsg}")

### to_crs / to_epsg — reproject mesh coordinates

In [ ]:
ds_wgs84 = ds.to_crs(4326)
print(f"Original bounds (EPSG:28992): {ds.bounds}")
print(f"Reprojected bounds (WGS 84): {ds_wgs84.bounds}")
print(f"New CRS: {ds_wgs84.epsg}")

## 3. No-Data Handling

### fillna — replace missing values

In [ ]:
filled = ds.fillna(-9999.0)
original = ds.get_data("depth").data
filled_arr = filled.get_data("depth").data
print(f"Original (has NaN): {original}")
print(f"Filled:             {filled_arr}")

### isnull / notnull — boolean masks

In [ ]:
null_mask = ds.isnull()
valid_mask = ds.notnull()
print(f"isnull:  {null_mask.get_data('depth').data}")
print(f"notnull: {valid_mask.get_data('depth').data}")
print(f"NaN count: {null_mask.get_data('depth').data.sum()}")

## 4. Value Transforms

### clip_values — clamp to a range

In [ ]:
clamped = ds.clip_values(2.0, 7.0)
print(f"Original: {ds.get_data('depth').data}")
print(f"Clamped:  {clamped.get_data('depth').data}")

### round — round to N decimals

In [ ]:
noisy = ds.fillna(0.0)
noisy = noisy.with_variable(
    "depth",
    noisy.get_data("depth").data + np.random.default_rng(42).normal(0, 0.1, 16),
    location="face",
)
rounded = noisy.round(1)
print(f"Noisy:   {noisy.get_data('depth').data[:6]}")
print(f"Rounded: {rounded.get_data('depth').data[:6]}")

### astype — cast to a different dtype

In [ ]:
filled_int = filled.astype(np.int32)
print(f"Original dtype: {filled.get_data('depth').data.dtype}")
print(f"Cast dtype:     {filled_int.get_data('depth').data.dtype}")
print(f"Values:         {filled_int.get_data('depth').data}")

### isin — membership test

In [ ]:
membership = ds.isin([1.0, 3.0, 5.0, 7.0, 9.0])
print(f"Values:    {ds.get_data('depth').data}")
print(f"Is in set: {membership.get_data('depth').data}")

## 5. Domain Metrics

### domain_area — total mesh area

In [ ]:
area = ds.domain_area()
print(f"Total mesh area: {area:.0f} square units")
print(f"Expected (400×400): {400 * 400}")

### count_domain_cells — number of active faces

In [ ]:
n_all = ds.count_domain_cells()
print(f"Total faces: {n_all}")
print(f"Expected: {(nx - 1) * (ny - 1)}")

## 6. Dataset Comparison

### equals — value equality (ignores dtype)

In [ ]:
ds_copy = ds.copy()
print(f"ds.equals(ds_copy): {ds.equals(ds_copy)}")

ds_modified = ds.fillna(0.0)
print(f"ds.equals(ds_modified): {ds.equals(ds_modified)}")

### identical — strict equality (dtype, CRS, nodata, topology)

In [ ]:
print(f"ds.identical(ds_copy): {ds.identical(ds_copy)}")

ds_int = ds.fillna(0.0).astype(np.int32)
ds_float = ds.fillna(0.0)
print(f"Same values, different dtype: {ds_int.identical(ds_float)}")

## 7. Point Extraction

### extract — sample values at specific coordinates

In [ ]:
query_x = np.array([50.0, 150.0, 250.0, 350.0])
query_y = np.array([50.0, 150.0, 250.0, 350.0])
values = ds.extract(query_x, query_y)
print("Extracted values at diagonal points:")
for xi, yi, vi in zip(query_x, query_y, values):
    print(f"  ({xi:.0f}, {yi:.0f}) → {vi}")

### sample_random — random spatial samples

In [ ]:
rx, ry, rvals = ds.sample_random(5, seed=42)
print("Random samples (n=5, seed=42):")
for xi, yi, vi in zip(rx, ry, rvals):
    print(f"  ({xi:.1f}, {yi:.1f}) → {vi}")

## 8. Graph Operations

### proximity — graph distance from target faces

In [ ]:
target = np.array([0])
dist = ds.proximity(target)
print("Distance from face 0 (top-left):")
print(dist.reshape(4, 4))

### sieve — remove small connected regions

In [ ]:
island_data = np.full(16, np.nan)
island_data[0] = 1.0
island_data[4:8] = [2.0, 3.0, 4.0, 5.0]
island_data[8:12] = [6.0, 7.0, 8.0, 9.0]

ds_island = ds.with_variable("depth", island_data, location="face")
print(f"Before sieve: {ds_island.get_data('depth').data}")

sieved = ds_island.sieve(threshold=3)
print(f"After sieve (threshold=3): {sieved.get_data('depth').data}")
print("(The isolated face 0 was blanked to NaN)")

## 9. Contouring

### contour — extract iso-lines as GeoDataFrame

In [ ]:
ds_clean = ds.fillna(5.0)
gdf = ds_clean.contour(levels=[2.0, 4.0, 6.0, 8.0])
print("Contour GeoDataFrame:")
print(gdf)
print(f"\nGeometry types: {gdf.geometry.geom_type.unique()}")

## 10. Plotting

### plot_histogram — value distribution

In [ ]:
fig, ax, hist = ds.plot_histogram(bins=8)
ax.set_title("Depth distribution")
ax.set_xlabel("Depth")
ax.set_ylabel("Count")

## 11. Serialization

### to_bytes — in-memory NetCDF round-trip

In [ ]:
blob = ds.to_bytes()
print(f"Serialized size: {len(blob)} bytes")

ds_roundtrip = UgridDataset.from_bytes(blob)
print(f"Round-trip equals: {ds.equals(ds_roundtrip)}")
print(f"Round-trip CRS:    {ds_roundtrip.epsg}")

## Summary

This notebook demonstrated 17 GIS-parity methods added to
`UgridDataset`:

| Category | Methods |
|---|---|
| CRS | `set_crs`, `to_crs`, `to_epsg` |
| No-data | `fillna`, `isnull`, `notnull` |
| Value transforms | `clip_values`, `round`, `astype`, `isin` |
| Domain metrics | `domain_area`, `count_domain_cells` |
| Comparison | `equals`, `identical` |
| Point extraction | `extract`, `sample_random` |
| Graph ops | `proximity`, `sieve` |
| Contouring | `contour` |
| Plotting | `plot_histogram` |
| I/O | `to_bytes` |

For the complete list of all ~95 methods available on
`UgridDataset` (including inherited mixin methods), see the
[Method Reference](../../../reference/netcdf/ugrid/api-methods.md).